# Part 10, notebook 5 of 5: what goes wrong, and the tests

Companion to *When One Agent Is Not Enough*. The five notebooks follow the post in order. Each explains the idea with a daily-life picture, shows the real source, and runs it. **Cells marked "Your turn" are for you to change and rerun.**

| # | Notebook | Post section | Needs the model server? |
|---|---|---|---|
| 10a | `10a_three_ways` | Three ways to spend model contexts | two short live cells |
| 10b | `10b_supervisor_and_handoff` | A supervisor and two specialists | live cells (about 1 minute) |
| 10c | `10c_fanout_and_panel` | One sub-agent per box, and a panel | live cells (about 30 seconds) |
| 10d | `10d_the_200_image_results` | The results | no (reads the saved runs) |
| 10e | `10e_failures_and_tests` | What goes wrong | no |

**The picture.** A relay race. A team of five runners is only as safe as the handovers: the baton can be dropped, passed to the wrong lane, or handed over with a different baton.
This notebook looks at the dropped batons that actually happened in the 2,000 runs, then at the offline test suite that makes each one happen on purpose.

In [1]:
import sys, json, os
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("..").resolve()))     # this notebook lives in literate/, the code in notebooks/agentic-zero-to-advanced/
OUT = Path("..").resolve() / "out"
os.environ.setdefault("PART10_MODEL", "qwen3.8-27b")   # the model name the local vLLM server serves
import sub_agents as sa
import sub_summary as ss
%matplotlib inline

## 1. The harness crashed on a string id (real)

The first full run lost one supervisor image: the model submitted a candidate id as a string, and the rules function compared it with an integer.
The failed line is still in the file as evidence. The fix is to **drop and count** ids that are not integers, never to guess what they meant.

In [2]:
bad = [json.loads(l) for l in (OUT / "sub_runs_sup.jsonl").read_text().splitlines()]
bad = [r for r in bad if r["status"] != "ok"]
print(bad)
print("\nafter the fix:", sa.rules_filter("test_00002", ["0", True, 1.5, None, 0]), " <- only the real integer survives")

[{'arm': 'supervisor', 'run': 0, 'image': 'test_00059', 'status': "error: TypeError: '<=' not supported between instances of 'int' and 'str'"}]

after the fix: [0]  <- only the real integer survives


## 2. Tool names the server could not parse (real)

In 13 supervisor tool calls the name that reached the loop was not a tool name at all: raw template text such as `<function=inspect_stage`, or `"\nsubmit_report"` with a stray newline. The loop answered "unknown tool" and the model tried again. It cost a step each time and nothing else, but it is the kind of thing that only shows up when you count tool names.

In [3]:
from collections import Counter
sup = ss.load()["supervisor"][0]
names = Counter(t for r in sup.values() for t in r["tools"])
print(names)

Counter({'inspect_stage': 207, 'submit_report': 200, 'judge_stage': 149, '<function=inspect_stage': 4, '\nsubmit_report': 3, '\n<function=judge_stage': 2, '\nsubmit_report\n<parameter=kept_ids': 1, '<function=submit_report': 1, '\n<function=inspect_stage': 1, '\njudge_stage': 1})


## 3. A failed sub-agent: drop the box or keep it?

With one batched judge, one malformed answer (after the retry) loses **every** box of the image. With a sub-agent per box it loses one, and the code decides what a lost box means. The fan-out here
keeps it (as uncertain), because a failure must never silently throw away a defect. The scripted fake model below makes both failures happen on purpose.

In [4]:
import types

class Garbage:
    """A fake model server whose every answer is the word 'garbage' (the test suite has a fuller scripted fake)."""
    def __init__(self):
        msg = types.SimpleNamespace(content="garbage", tool_calls=None)
        resp = types.SimpleNamespace(choices=[types.SimpleNamespace(message=msg)], usage=types.SimpleNamespace(prompt_tokens=100, completion_tokens=2))
        self.chat = types.SimpleNamespace(completions=types.SimpleNamespace(create=lambda **kw: resp))

IMG = "test_00002"
sa.CLIENT = Garbage()
print("batched, model always answers garbage ->", {k: sa.arm_batched(IMG)[k] for k in ("kept", "judge_failed", "calls")})
print("fanout,  model always answers garbage ->", {k: sa.arm_fanout(IMG)[k] for k in ("kept", "judge_failed", "calls")}, " (every box kept: nothing silently dropped)")
sa.CLIENT = None

batched, model always answers garbage -> {'kept': [], 'judge_failed': 1, 'calls': 2}
fanout,  model always answers garbage -> {'kept': [0, 1, 2], 'judge_failed': 3, 'calls': 6}  (every box kept: nothing silently dropped)


## 4. The test suite

29 checks, all offline, a scripted fake model instead of the server: the loop (nudge, bad JSON, unknown tool, step limit), the rules, the batched retry, every fan-out and panel path, the
supervisor's hand-off, and the id handling above.

In [5]:
import subprocess
r = subprocess.run([sys.executable, "test_sub_paths.py"], cwd=str(Path("..").resolve()), capture_output=True, text=True)
print(r.stdout or r.stderr)

ok  loop: finishes on submit
ok  loop: unknown tool, bad JSON and tool error all go back to the model, none crash it
ok  loop: a talking model is nudged
ok  meter: counts every call, by role
ok  loop: stops at the step limit with nothing submitted
ok  rules: duplicates and out-of-range ids are dropped
ok  rules: ids that are not integers (strings, booleans, floats) are dropped, not guessed
ok  finish: the dropped ids are recorded
ok  batched: one call, kept = defects after rules
ok  batched: a malformed answer is retried with the error
ok  batched: valid JSON that skips ids is refused, and one failure drops the WHOLE image
ok  fanout: one call per box
ok  fanout: every context is just system + one box
ok  fanout: each sub-agent saw exactly its own box, once
ok  fanout: the box is in its own brief
ok  fanout: a failed sub-agent keeps its box (never silently drops it) and does not touch the other boxes
ok  fanout: the failed box was retried once
ok  panel3: three sub-agents per box
ok  p